In [1]:
import pandas as pd
import numpy as np
import json
import os
from collections import defaultdict

In [ ]:
data_predict_result_path = '/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label'
adr_name_list = ['Blood and lymphatic system disorders','Cardiac disorders','Ear and labyrinth disorders','Endocrine disorder',
                 'Eye disorders','Gastrointestinal disorders','Hepatobiliary disorders','Immune system disorders','Infections and infestations',
                 'Metabolism and nutrition disorders','Musculoskeletal and connective tissue disorders','Nervous system disorders',
                 'Psychiatric disorders','Renal and urinary disorders','Reproductive system and breast disorders',
                 'Respiratory, thoracic and mediastinal disorders','Skin and subcutaneous tissue disorders','Vascular disorders'] #'Mutagenicity',


#### 1. Analysis of PreMOTA off-target Prediction Results

In [ ]:
for adr_name in adr_name_list:
    print("now parsing is: ", adr_name)
    predict_dict_path = os.path.join(data_predict_result_path, 'alladrdrug_dup_predict_result.json') # {smiles1:{target1: [pK, pAC50], target2: [pK, pAC50]...},smiles2:{target1: [pK, pAC50], target2: [pK, pAC50]...}}
    data_path = os.path.join(data_predict_result_path, adr_name, 'adrdrug_dup_label_filter.csv') # Drug,smiles_uncanical,smiles,label

    data_df = pd.read_csv(data_path) # canonical_smiles、label
    adr_smiles_list = data_df['smiles'].tolist()
    print("data_df.shape: ", data_df.shape) 
    # 构建canonical_smiles和label的字典
    label_dict = dict(zip(data_df['smiles'], data_df['label']))
    print("len(label_dict): ", len(label_dict))
   
    with open(predict_dict_path, 'r') as f:
        predict_dict = json.load(f)

    print("len(predict_dict): ", len(predict_dict))
    
    predict_dict_new = defaultdict(dict)
    for key,value in predict_dict.items():
        if key in adr_smiles_list:
            for k,v in value.items():
                value1 = v[0] # pK obtained by converting the original nM to M and then taking -log10
                value2 = v[1] # pAC50
                value1_uM = 1e6*10**(-value1)
                value2_uM = 1e6*10**(-value2)
                values_new = (value1_uM+value2_uM)/2
                predict_dict_new[key][k] = value2_uM
                # predict_dict_new[key][k] = value2

    aff_df = pd.DataFrame(predict_dict_new).T
    aff_df.index.name = 'smiles'
    aff_df.reset_index(inplace=True)
    aff_df['label'] = aff_df['smiles'].map(label_dict)

    print("df.shape: ", aff_df.shape)
    print("parse done!!!")

    aff_df.to_csv(os.path.join(data_predict_result_path, adr_name, 'adr_label_offtarget_predict.csv'), index=False)

now parsing is:  Blood and lymphatic system disorders
data_df.shape:  (4955, 4)
len(label_dict):  4955
len(predict_dict):  9202
df.shape:  (4955, 196)
parse done!!!
now parsing is:  Cardiac disorders
data_df.shape:  (4615, 4)
len(label_dict):  4615
len(predict_dict):  9202
df.shape:  (4615, 196)
parse done!!!
now parsing is:  Ear and labyrinth disorders
data_df.shape:  (4436, 4)
len(label_dict):  4436
len(predict_dict):  9202
df.shape:  (4436, 196)
parse done!!!
now parsing is:  Endocrine disorder
data_df.shape:  (3605, 4)
len(label_dict):  3605
len(predict_dict):  9202
df.shape:  (3605, 196)
parse done!!!
now parsing is:  Eye disorders
data_df.shape:  (3492, 4)
len(label_dict):  3492
len(predict_dict):  9202
df.shape:  (3492, 196)
parse done!!!
now parsing is:  Gastrointestinal disorders
data_df.shape:  (4851, 4)
len(label_dict):  4851
len(predict_dict):  9202
df.shape:  (4851, 196)
parse done!!!
now parsing is:  Hepatobiliary disorders
data_df.shape:  (5201, 4)
len(label_dict):  5201

#### 2. Analysis of MotifAttnNet Prediction Results

In [ ]:
import pandas as pd
df = pd.read_csv('/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label/Blood and lymphatic system disorders/adr_label_offtarget_predict.csv')
print(df.shape)
offtarget_list = df.columns.tolist()
offtarget_list = offtarget_list[1:-1]
print(len(offtarget_list))
df.head(3)

(4955, 196)
194


,smiles,P35414,P24530,P30518,P35346,Q9Y271,P34998,P30874,P34995,P08173,...,P31645,Q01959,P31652,P23977,Q9GZV3,P28572,P30536,P01375,Q99720,label
0,C=C(CC)C(=O)c1ccc(OCC(=O)O)c(Cl)c1Cl,3.546367,11.121163,0.067291,0.120398,4.223025,0.628767,2.227337,0.720610,0.289470,...,1.866084,75.317584,0.214716,18.365018,17.194091,2.194313,835.050760,1.126558,0.311167,1
1,CC1(C)O[C@@H]2C[C@H]3[C@@H]4C[C@H](F)C5=CC(=O)...,0.975592,7.151165,1.567875,0.377504,10.191785,0.724041,10.952628,9.724267,0.321699,...,0.045250,1.158085,0.231008,1.030109,3.040490,0.216013,0.041167,0.336765,8.112931,0
2,COc1cc(C(=O)NC2CCN(C)CC2)c(F)cc1Nc1ncc(C(F)(F)...,0.088585,4.182401,1.454840,0.047998,0.102483,0.264737,0.005746,0.108122,0.157425,...,0.350535,1.028661,0.182644,0.624699,1.111731,0.094116,0.179618,0.899578,38.160078,0


In [ ]:
# Train HetSia-SafeNet to select the Cmax,free predictive value of the drug at a dose of 100mg
import pathlib
data_predict_result_path_old = '/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label/'
data_cmax_path = '/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label/dose_study/drug_cmax_predict_100mg.csv' #smiles、cmax_free(uM)、Drug
# Extract the corresponding target prediction value from each adr_term
for adr_name in adr_name_list:
    offtarget_predict_result_df = pd.read_csv(os.path.join(data_predict_result_path,adr_name,'adr_label_offtarget_predict.csv')) #smiles
    
    # Retrieve the Cmax and free data
    drug_cmax_df = pd.read_csv(data_cmax_path)
    drug_name_list = drug_cmax_df['Drug'].tolist()
    drugname_smiles_dict = dict(zip(drug_cmax_df['smiles'], drug_cmax_df['Drug']))
    drugname_cmax_dict = dict(zip(drug_cmax_df['smiles'], drug_cmax_df['cmax_free(uM)']))

    drugname_smiles_list = drug_cmax_df['smiles'].tolist()
    
    # Select the rows in offtarget_predict_result_df where smiles is included in drugname_smiles_list
    offtarget_predict_result_df_choose = offtarget_predict_result_df[offtarget_predict_result_df['smiles'].isin(drugname_smiles_list)]
    offtarget_predict_result_df_choose.insert(0, 'Drug', offtarget_predict_result_df_choose['smiles'].map(drugname_smiles_dict))
    offtarget_predict_result_df_choose.insert(1, 'cmax_free(uM)', offtarget_predict_result_df_choose['smiles'].map(drugname_cmax_dict))

    feature_list = offtarget_predict_result_df.columns.tolist()[1:-1]
    
    feature_list.insert(0, 'Drug')
    feature_list.insert(1, 'smiles')
    feature_list.insert(2, 'label')
    feature_list.insert(3, 'cmax_free(uM)')

    filter_offtarget_predict_result_df_choose = offtarget_predict_result_df_choose[feature_list]
    print(filter_offtarget_predict_result_df_choose['label'].value_counts())
    
    save_path = os.path.join('/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label/',adr_name)
    pathlib.Path(save_path).mkdir(parents=True, exist_ok=True)
    filter_offtarget_predict_result_df_choose.to_csv(os.path.join(save_path,'adr_label_offtarget_cmax_predict.csv'), index=False)

label
0    2930
1    2025
Name: count, dtype: int64
label
0    2392
1    2223
Name: count, dtype: int64
label
0    3039
1    1397
Name: count, dtype: int64
label
0    2676
1     929
Name: count, dtype: int64
label
1    1943
0    1549
Name: count, dtype: int64
label
1    2846
0    2005
Name: count, dtype: int64
label
0    3451
1    1750
Name: count, dtype: int64
label
1    2157
0    2093
Name: count, dtype: int64
label
0    2500
1    2085
Name: count, dtype: int64
label
0    2465
1    2141
Name: count, dtype: int64
label
0    2486
1    2155
Name: count, dtype: int64
label
1    2905
0    2221
Name: count, dtype: int64
label
0    3208
1    2226
Name: count, dtype: int64
label
0    2634
1    2061
Name: count, dtype: int64
label
0    2789
1    1525
Name: count, dtype: int64
label
1    2345
0    1808
Name: count, dtype: int64
label
1    2861
0    2325
Name: count, dtype: int64
label
0    3024
1    2456
Name: count, dtype: int64


#### 3.Construction of the dataset required for HetSia-SafeNet training

In [ ]:
import pandas as pd
import os
import numpy as np
adr_name_list = ['Blood and lymphatic system disorders','Cardiac disorders','Ear and labyrinth disorders','Endocrine disorder',
                 'Eye disorders','Gastrointestinal disorders','Hepatobiliary disorders','Immune system disorders','Infections and infestations',
                 'Metabolism and nutrition disorders','Musculoskeletal and connective tissue disorders','Nervous system disorders',
                 'Psychiatric disorders','Renal and urinary disorders','Reproductive system and breast disorders',
                 'Respiratory, thoracic and mediastinal disorders','Skin and subcutaneous tissue disorders','Vascular disorders']

data_predict_result_path = "/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label/"
for adr_name in adr_name_list:
    adr_name_df = pd.read_csv(os.path.join(data_predict_result_path,adr_name,'adr_label_offtarget_cmax_predict.csv'))
    # Perform an np.log10 transformation on the data after the fourth column
    print("挑选值前尺寸：", adr_name_df.shape)
    adr_name_df = adr_name_df[adr_name_df['cmax_free(uM)']>0]
    # Delete the rows where the label column is empty
    adr_name_df = adr_name_df[adr_name_df['label'].notna()]
    print("挑选值后尺寸：", adr_name_df.shape)
    adr_name_df.iloc[:,3:] = np.log10(adr_name_df.iloc[:,3:])

    adr_name_df_cmax = adr_name_df.copy()
    new_columns = {adr_name_df_cmax.columns[i] + '*Cmax': adr_name_df_cmax[adr_name_df_cmax.columns[i]] * adr_name_df_cmax['cmax_free(uM)'] for i in range(4,len(adr_name_df_cmax.columns))}
    # Add the new column to the original data frame all at once
    adr_name_df_cmax = pd.concat([adr_name_df_cmax, pd.DataFrame(new_columns)], axis=1)
    columns_list = adr_name_df_cmax.columns.tolist()
    # Select the columns_list that contains the value of cmax/Cmax
    columns_list_choose = []
    for i in range(len(columns_list)):
        if 'cmax' in columns_list[i] or 'Cmax' in columns_list[i] or 'Drug' in columns_list[i] or 'smiles' in columns_list[i] or 'label' in columns_list[i]:
            columns_list_choose.append(columns_list[i])
    adr_name_df_cmax_del = adr_name_df_cmax[columns_list_choose]

    
    adr_name_df = adr_name_df.drop(columns=['cmax_free(uM)'])
    adr_name_df.to_csv(os.path.join(data_predict_result_path,adr_name,'adr_ac50_drug_data_log10.csv'),index=False)
    adr_name_df_cmax_del.to_csv(os.path.join(data_predict_result_path,adr_name,'adr_cmax_drug_data_log10.csv'),index=False)
    adr_name_df_cmax.to_csv(os.path.join(data_predict_result_path,adr_name,'adr_ac50_cmax_drug_data_log10.csv'),index=False)

挑选值前尺寸： (4955, 198)
挑选值后尺寸： (4955, 198)
挑选值前尺寸： (4615, 198)
挑选值后尺寸： (4615, 198)
挑选值前尺寸： (4436, 198)
挑选值后尺寸： (4436, 198)
挑选值前尺寸： (3605, 198)
挑选值后尺寸： (3605, 198)
挑选值前尺寸： (3492, 198)
挑选值后尺寸： (3492, 198)
挑选值前尺寸： (4851, 198)
挑选值后尺寸： (4851, 198)
挑选值前尺寸： (5201, 198)
挑选值后尺寸： (5201, 198)
挑选值前尺寸： (4250, 198)
挑选值后尺寸： (4250, 198)
挑选值前尺寸： (4585, 198)
挑选值后尺寸： (4585, 198)
挑选值前尺寸： (4606, 198)
挑选值后尺寸： (4606, 198)
挑选值前尺寸： (4641, 198)
挑选值后尺寸： (4641, 198)
挑选值前尺寸： (5126, 198)
挑选值后尺寸： (5126, 198)
挑选值前尺寸： (5434, 198)
挑选值后尺寸： (5434, 198)
挑选值前尺寸： (4695, 198)
挑选值后尺寸： (4695, 198)
挑选值前尺寸： (4314, 198)
挑选值后尺寸： (4314, 198)
挑选值前尺寸： (4153, 198)
挑选值后尺寸： (4153, 198)
挑选值前尺寸： (5186, 198)
挑选值后尺寸： (5186, 198)
挑选值前尺寸： (5480, 198)
挑选值后尺寸： (5480, 198)


##### 3.1 Obtain the overall representation matrix

In [ ]:
# Obtain the overall representation matrix
train_df = pd.DataFrame()
save_path = "/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label/"
for adr_dataname in adr_name_list:
    print(adr_dataname)
    
    adr_df = pd.read_csv(os.path.join(save_path,adr_dataname,'adr_ac50_cmax_drug_data_log10.csv'))
    adr_df.drop_duplicates(subset=['smiles'],inplace=True)
    
    train_df = pd.concat([train_df,adr_df],axis=0)
    train_df.drop_duplicates(subset=['smiles'],inplace=True)


save_path_o = os.path.join(save_path,"ADR_multitask_dataset","single_task")
os.makedirs(save_path_o,exist_ok=True)
# Remove the "label" column because it has been shuffled after deduplication and is meaningless. Different drugs have different labels under different ADR categories
train_df.drop(columns=['label'],inplace=True)
print("finally train_df.shape",train_df.shape)
train_df.to_csv(os.path.join(save_path_o,"all_data.csv"),index=False)



Blood and lymphatic system disorders
Cardiac disorders
Ear and labyrinth disorders
Endocrine disorder
Eye disorders
Gastrointestinal disorders
Hepatobiliary disorders
Immune system disorders
Infections and infestations
Metabolism and nutrition disorders
Musculoskeletal and connective tissue disorders
Nervous system disorders
Psychiatric disorders
Renal and urinary disorders
Reproductive system and breast disorders
Respiratory, thoracic and mediastinal disorders
Skin and subcutaneous tissue disorders
Vascular disorders
finally train_df.shape (9202, 391)


##### 3.2 Obtain the total label matrix

In [ ]:
# Obtain the total label matrix
multitask_df = pd.DataFrame(columns=["smiles"])
save_path = "/home/datahouse1/liujin/HetSia-SafeNet/data_process/ADR_tree_label/"
for adr_dataname in adr_name_list:
    print(adr_dataname)
    adr_df = pd.read_csv(os.path.join(save_path,adr_dataname,'adr_ac50_cmax_drug_data_log10.csv'))
    adr_df = adr_df[["smiles","label"]]
    adr_df.drop_duplicates(subset=['smiles'],inplace=True)
    # 更改label列名为adr_dataname
    adr_df.rename(columns={"label":adr_dataname},inplace=True)
    # 按照smiles列合并，没有值的地方添空值

    multitask_df = pd.merge(multitask_df,adr_df,on="smiles",how='outer')


save_path_o = os.path.join(save_path,"ADR_multitask_dataset")
os.makedirs(save_path_o,exist_ok=True)
print("finally train_df.shape",multitask_df.shape)
multitask_df.to_csv(os.path.join(save_path_o,"mutiltask_adr_data.csv"),index=False) # Compounds with cmax prediction less than 0 have been removed

Blood and lymphatic system disorders
Cardiac disorders
Ear and labyrinth disorders
Endocrine disorder
Eye disorders
Gastrointestinal disorders
Hepatobiliary disorders
Immune system disorders
Infections and infestations
Metabolism and nutrition disorders
Musculoskeletal and connective tissue disorders
Nervous system disorders
Psychiatric disorders
Renal and urinary disorders
Reproductive system and breast disorders
Respiratory, thoracic and mediastinal disorders
Skin and subcutaneous tissue disorders
Vascular disorders
finally train_df.shape (9202, 19)
